# Computer Vision - Part 2

### Lab 1: Capture Frames from Camera

The starting point for all vision work — opening a camera device, reading frames, and displaying them. This forms the basic I/O loop used in every real-time vision application.

Key Concepts

- `VideoCapture(0)`: opens the first available camera (use 1, 2 for additional cameras)

- `cap.read()`: returns (success_flag, frame) — always check `ret` before using `frame`

- `frame`: a NumPy array of shape (H, W, 3) in BGR color format

- For continuous video, wrap in a `while True` loop and check for key press

In [1]:
!pip install opencv-python
!pip install pickle


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import cv2

cap = cv2.VideoCapture(0)
ret, frame = cap.read()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow('Frame', frame)
    # Press 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### Lab 2: Construct the K Matrix

Before calibrating, understand how to manually construct `K` from known or estimated parameters. This is also useful when you have a camera spec sheet with focal length in **mm**.

Practical Notes

- For a **640×480 image**, a reasonable starting estimate is `cx=320`, `cy=240`
- `fx ≈ fy` for most cameras (square pixels)
- A typical webcam has `fx ≈ 600–800 px`; a phone camera may have `fx > 1000 px`
- Always validate `K` against actual calibration — manual estimates are rough

In [3]:
import numpy as np

fx, fy = 800, 800  # pixels
cx, cy = 320, 240  # image center

K = np.array([[fx, 0, cx], 
              [0, fy, cy], 
              [0, 0, 1]], dtype=float)

print("Matrix K:\n", K)

Matrix K:
 [[800.   0. 320.]
 [  0. 800. 240.]
 [  0.   0.   1.]]


### Lab 3: Project a 3D Point to 2D

The core operation: given a 3D point in camera coordinates, compute its pixel location using the camera matrix `K`.

Understanding the Steps
- Divide `X`, `Y` by `Z`: this is the perspective divide giving normalized coordinates
- Multiply by `K`: converts to pixel coordinates `(u, v)`
- The point `(1, 2, 5)` is 1m right, 2m down, and 5m away from the
camera

In [4]:
import numpy as np

X_3d = np.array([1.0, 2.0, 5.0])
# Perspective divide first
x_norm = X_3d[0] / X_3d[2]
y_norm = X_3d[1] / X_3d[2]

p_h = np.array([x_norm, y_norm, 1])
p_2d = K @ p_h

u = int(p_2d[0])
v = int(p_2d[1])

print("p_2d = K @ p_h:\n", p_2d)
print("u = p_2d[0]:", u)
print("v = p_2d[1]:", v)

p_2d = K @ p_h:
 [480. 560.   1.]
u = p_2d[0]: 480
v = p_2d[1]: 560


- Try changing `Z` — notice how `(u, v)` changes as the object moves closer/farther

In [5]:
import numpy as np

x, y, z = 1.0, 2.0, 6.5
X_3d_new = np.array([x, y, z])
# Perspective divide first
x_norm = X_3d_new[0] / X_3d_new[2]
y_norm = X_3d_new[1] / X_3d_new[2]

p_h = np.array([x_norm, y_norm, 1])
p_2d = K @ p_h

u = int(p_2d[0])
v = int(p_2d[1])

print("p_2d = K @ p_h:\n", p_2d)
print("u = p_2d[0]:", u)
print("v = p_2d[1]:", v)

p_2d = K @ p_h:
 [443.07692308 486.15384615   1.        ]
u = p_2d[0]: 443
v = p_2d[1]: 486


### Lab 4: Load Calibration Data

In production robotics pipelines, calibration data is calibrated once and saved to disk, then loaded at runtime. This avoids repeating calibration every session.

Best Practices
- Save `K`, `dist`, `image_size`, and `reprojection error` together in one
file
- Use `pickle` for Python objects or `np.savez()` for NumPy arrays
- Also consider saving `rvecs`/`tvecs` if you need pose data later
- Recalibrate when the camera lens is changed, refocused, or physically adjusted

In [6]:
import pickle
import numpy as np

K, dist, image_size, reprojection_error = (np.array([[800, 0, 320], 
                                                   [0, 800, 240], 
                                                   [0, 0, 1]], dtype=float),
                                           np.array([0.1, -0.05, 0.001, 0.0005, 0.0]),
                                           (640, 480),
                                           0.5)
# Save calibration data
calib_data = {'K': K, 
              'dist': dist, 
              'image_size': image_size, 
              'reprojection_error': reprojection_error}

with open('calib.pkl', 'wb') as f:
    pickle.dump(calib_data, f)

# Load saved calibration
with open('calib.pkl', 'rb') as f:
    calib_data = pickle.load(f)

K = calib_data['K']
dist = calib_data['dist']

print("K matrix loaded:\n", K)

# Save rvecs/tvecs
rvecs = [np.array([[0.1], 
                   [0.2], 
                   [0.3]])]

tvecs = [np.array([[1.0], 
                   [2.0], 
                   [3.0]])]

pose_data = {'rvecs': rvecs, 'tvecs': tvecs}

with open('pose.pkl', 'wb') as f:
    pickle.dump(pose_data, f)

K matrix loaded:
 [[800.   0. 320.]
 [  0. 800. 240.]
 [  0.   0.   1.]]


### Lab 5: Undistort an Image

Apply the estimated distortion model to produce a geometrically corrected image — a prerequisite for any accurate measurement or 3D reconstruction.

What to Observe
- With strong barrel distortion (wide-angle lens), straight lines near the edges become visibly straighter after undistortion
- The image may show black borders at the edges — this is expected and can be cropped
- For video, precompute `mapx`, `mapy` once and reuse `cv2.remap()` for each frame

In [7]:
import cv2
import pickle

# Load calibration
with open('calib.pkl', 'rb') as f:
    calib_data = pickle.load(f)

K = calib_data['K']
dist = calib_data['dist']

# Open camera
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

# Read one frame to get size
ret, frame = cap.read()
if not ret:
    print("Cannot read frame")
    exit()

h, w = frame.shape[:2]

# Precompute remap (FAST)
mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K, (w, h), cv2.CV_32FC1)

# Loop
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Undistort
    undist1 = cv2.undistort(frame, K, dist)
    undist2 = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)

    # Show
    cv2.imshow('Original', frame)
    cv2.imshow('Undistorted', undist1)
    cv2.imshow('Undistorted (remap)', undist2)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()

### Lab 6: Draw Projected Points on Image

Visualize the result of 3D projection by drawing projected points directly onto camera frames. This is the standard technique for verifying calibration accuracy.

Visualization Tips
- Use green circles for projected points and red for detected points — compare them to assess calibration quality
- Use `cv2.putText()` to label each point with its 3D coordinates for debugging
- The distance between projected and detected points is the reprojection error
- Good calibration: `reprojection error < 1 pixel`

In [ ]:
img = frame.copy()

u1, v1 = 400, 300
u2, v2 = 450, 350

projected_points = [(u1, v1), (u2, v2)]
print("Projected points (u, v):", projected_points)

for (u, v) in projected_points:
    cv2.circle(img, (u, v), 5, (0, 255, 0), -1)

cv2.imshow('Projected', img)

key = cv2.waitKey(0) & 0xFF
if key == ord('q'):
    cv2.destroyAllWindows()

Projected points (u, v): [(400, 300), (450, 350)]


: 

### Lab 7: Determine Object Grasp Position

The ultimate goal in robot vision: detect an object in the image and determine its 3D position for the robot arm to grasp. This combines all previous concepts.

1. Detect Object in Image

Use color segmentation, thresholding, or a detector to find the object's pixel region in the 2D image frame.

In [3]:
import cv2

img = cv2.imread("pink_flower.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Threshold first
_, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# Find contours
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Draw contours
for cnt in contours:
    cv2.drawContours(img, [cnt], -1, (0,255,0), 2)

cv2.imshow("Contours", img)

# Save the image with contours
cv2.imwrite("pink_flower_contours.jpg", img)

key = cv2.waitKey(0) & 0xFF
if key == ord('q'):
    cv2.destroyAllWindows()

2. Compute 2D Centroid (u_obj, v_obj)

Calculate the center of the detected object mask using moments: `cx = M10/M00`, `cy = M01/M00`.

In [4]:
# Use the same contours to compute moments
for cnt in contours:
    M = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        print(f"Centroid: ({cx}, {cy})")
    else:
        print("Contour area is zero, cannot compute centroid.")

Contour area is zero, cannot compute centroid.
Contour area is zero, cannot compute centroid.
Centroid: (344, 2822)
Contour area is zero, cannot compute centroid.
Centroid: (249, 2822)
Contour area is zero, cannot compute centroid.
Centroid: (220, 2814)
Centroid: (224, 2808)
Contour area is zero, cannot compute centroid.
Centroid: (322, 2798)
Centroid: (334, 2799)
Centroid: (346, 2795)
Contour area is zero, cannot compute centroid.
Centroid: (336, 2789)
Contour area is zero, cannot compute centroid.
Contour area is zero, cannot compute centroid.
Centroid: (352, 2782)
Centroid: (318, 2777)
Contour area is zero, cannot compute centroid.
Contour area is zero, cannot compute centroid.
Contour area is zero, cannot compute centroid.
Contour area is zero, cannot compute centroid.
Centroid: (474, 2772)
Contour area is zero, cannot compute centroid.
Centroid: (422, 2802)
Contour area is zero, cannot compute centroid.
Contour area is zero, cannot compute centroid.
Centroid: (1843, 2734)
Centroid

3. Convert to 3D (requires depth)

Using a depth camera (RGBD) or stereo vision, retrieve the depth `Z` at `(u, v)`. Then: `X = (u - cx) * Z / fx`, `Y = (v - cy) * Z / fy`.

In [ ]:
# Use the same contours to compute bounding boxes to convert to 3d
import numpy as np

# Simulated depth map
depth_map = np.random.uniform(0.5, 5.0, size=gray.shape).astype(np.float32) 

# Example camera intrinsics
fx, fy = 600, 600
cx, cy = 320, 240

for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)

    # Center pixel
    u = x + w // 2
    v = y + h // 2

    # Get depth (Z)
    Z = depth_map[v, u]  # depth in meters (or mm → convert!)

    if Z == 0:
        continue  # skip invalid depth

    # Convert to 3D
    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy

    print(f"3D Position: X={X:.3f}, Y={Y:.3f}, Z={Z:.3f}")

3D Position: X=0.094, Y=7.568, Z=1.757
3D Position: X=-0.346, Y=14.434, Z=3.352
3D Position: X=0.172, Y=17.730, Z=4.118
3D Position: X=-0.420, Y=16.701, Z=3.880
3D Position: X=-0.569, Y=20.994, Z=4.877
3D Position: X=-0.464, Y=17.823, Z=4.151
3D Position: X=-0.413, Y=10.623, Z=2.475
3D Position: X=-0.335, Y=9.154, Z=2.138
3D Position: X=0.045, Y=16.474, Z=3.851
3D Position: X=0.017, Y=14.497, Z=3.398
3D Position: X=0.087, Y=13.874, Z=3.250
3D Position: X=0.045, Y=4.220, Z=0.991
3D Position: X=0.000, Y=19.817, Z=4.661
3D Position: X=0.038, Y=5.318, Z=1.251
3D Position: X=0.241, Y=15.744, Z=3.707
3D Position: X=0.179, Y=11.410, Z=2.690
3D Position: X=0.069, Y=5.322, Z=1.256
3D Position: X=-0.007, Y=9.186, Z=2.172
3D Position: X=0.217, Y=3.421, Z=0.809
3D Position: X=0.665, Y=11.324, Z=2.679
3D Position: X=0.003, Y=7.823, Z=1.852
3D Position: X=0.242, Y=3.830, Z=0.908
3D Position: X=1.277, Y=20.874, Z=4.944
3D Position: X=0.585, Y=9.299, Z=2.206
3D Position: X=0.175, Y=4.430, Z=1.039
3D P

4. Output Robot Target `(X, Y, Z)`

The resulting 3D point in camera frame is transformed to the robot base frame via the hand-eye transform, giving the grasp target position.

In [6]:
import numpy as np

# Define the transformation matrix
r11, r12, r13 = 1, 0, 0
r21, r22, r23 = 0, 1, 0
r31, r32, r33 = 0, 0, 1
tx, ty, tz = 0.5, 0.0, 0.2

T_base_camera = np.array([[r11, r12, r13, tx], 
                          [r21, r22, r23, ty], 
                          [r31, r32, r33, tz], 
                          [0,   0,   0,   1 ]])

P_camera = np.array([X, Y, Z, 1])
P_base = T_base_camera @ P_camera
Xb, Yb, Zb = P_base[:3]

for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)

    # Center pixel
    u = x + w // 2
    v = y + h // 2

    Z = depth_map[v, u]
    if Z <= 0:
        continue

    # Camera intrinsics
    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy

    # Camera frame point
    P_camera = np.array([X, Y, Z, 1])

    # Transform to robot base
    P_base = T_base_camera @ P_camera

    Xb, Yb, Zb = P_base[:3]

    print(f"Robot Target: X={Xb:.3f}, Y={Yb:.3f}, Z={Zb:.3f}")

Robot Target: X=0.594, Y=7.568, Z=1.957
Robot Target: X=0.154, Y=14.434, Z=3.552
Robot Target: X=0.672, Y=17.730, Z=4.318
Robot Target: X=0.080, Y=16.701, Z=4.080
Robot Target: X=-0.069, Y=20.994, Z=5.077
Robot Target: X=0.036, Y=17.823, Z=4.351
Robot Target: X=0.087, Y=10.623, Z=2.675
Robot Target: X=0.165, Y=9.154, Z=2.338
Robot Target: X=0.545, Y=16.474, Z=4.051
Robot Target: X=0.517, Y=14.497, Z=3.598
Robot Target: X=0.587, Y=13.874, Z=3.450
Robot Target: X=0.545, Y=4.220, Z=1.191
Robot Target: X=0.500, Y=19.817, Z=4.861
Robot Target: X=0.538, Y=5.318, Z=1.451
Robot Target: X=0.741, Y=15.744, Z=3.907
Robot Target: X=0.679, Y=11.410, Z=2.890
Robot Target: X=0.569, Y=5.322, Z=1.456
Robot Target: X=0.493, Y=9.186, Z=2.372
Robot Target: X=0.717, Y=3.421, Z=1.009
Robot Target: X=1.165, Y=11.324, Z=2.879
Robot Target: X=0.503, Y=7.823, Z=2.052
Robot Target: X=0.742, Y=3.830, Z=1.108
Robot Target: X=1.777, Y=20.874, Z=5.144
Robot Target: X=1.085, Y=9.299, Z=2.406
Robot Target: X=0.675, Y=

### Lab 8: Estimate Extrinsic [R|t] with solvePnP

`cv2.solvePnP()` solves the Perspective-n-Point problem: given `N` known 3D points and their 2D projections, estimate the camera pose (`R` and `t`).

Key Notes
- `rvec`: rotation as a compact 3D vector (Rodrigues form) — not a
matrix!
- `cv2.Rodrigues()`: converts rvec 3×3 rotation matrix `R`
- Requires at least 4 non-coplanar point correspondences
- Used heavily in AR marker tracking and robot pose estimation
- Result: camera position and orientation relative to the calibration object

In [ ]:
import cv2

u1, v1 = 400, 300
u2, v2 = 450, 350
K = np.array([[800, 0, 320], 
              [0, 800, 240], 
              [0, 0, 1]], dtype=float)
dist = np.array([0.1, -0.05, 0.001, 0.0005, 0.0])

# Use the same contours to compute 3D pose (PnP)
objpoints = np.array([[0, 0, 0], 
                      [1, 0, 0], 
                      [1, 1, 0], 
                      [0, 1, 0]], dtype=np.float32)

imgpoints = np.array([[u1, v1], 
                      [u2, v2], 
                      [u2+50, v2], 
                      [u1+50, v1]], dtype=np.float32)

# cv2.solvePnP(# Nx3 world points, # Nx2 image points, # camera matrix, # distortion coeffs)
ret, rvec, tvec = cv2.solvePnP(objpoints, imgpoints, K, dist)
if ret:
    print("Rotation Vector (rvec):\n", rvec)
    print("Translation Vector (tvec):\n", tvec)
else:
    print("PnP failed to find a solution.")

Rotation Vector (rvec):
 [[ 1.60228546]
 [ 1.13665664]
 [-0.19103111]]
Translation Vector (tvec):
 [[ 1.01758833]
 [ 0.75279352]
 [10.18649454]]


: 